<a href="https://colab.research.google.com/github/Nevermind-ji/amazon_ml_challenge/blob/main/amazon_ml_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

# Path inside your Google Drive
folder_path = '/content/drive/MyDrive/Amazon_ML_Challenge/'

df1 = pd.read_csv(folder_path + 'train_source1.tsv', sep='\t')
df2 = pd.read_csv(folder_path + 'train_source2.tsv', sep='\t')
df3 = pd.read_csv(folder_path + 'train_source3.tsv', sep='\t')
gt  = pd.read_csv(folder_path + 'train_ground_truth.tsv', sep='\t')
import matplotlib.pyplot as plt


In [ ]:

df1

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India
...,...,...,...,...
2206816,S1-479751630,Arlyn Farooqi Prairie Medical,"16804 Stevenage Street, Surprise, AZ",US
2206817,S1-392755726,Patriot Fund,"MD, 7709 Ashdale Road, Capitol Heights",US
2206818,S1-716147977,Sloan Fact of Yonkers,"28 Armstrong Avenue, Unit Apartment 2, Yonkers...",US
2206819,S1-770136229,Ritter's Machine Works,"5609 Spring Meadow Road, Austin, TX",US


In [ ]:

df2

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
...,...,...,...,...
5034611,S2-147379319,CHOICE SECURITY PARTNERS CO,"IL, 023 Franklin St, DANVILLE",US
5034612,S2-828699622,TMQ Public (Center),"65 Fifth Street, FULTON, NY",US
5034613,S2-726846711,CONTINENTAL CDT PC,"SPENCER, MA, 18 GROVE ST",US
5034614,S2-115058866,लक्ष्मी मीडिया प्राइवेट लिमिटेड,"154, BHUGAON, PUNE, महाराष्ट्र",India


In [ ]:

df3

,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India
...,...,...,...,...
5285598,S3-549653960,Vip Solutions,NaN,India
5285599,S3-45212707,Cornerstone Nippon,"Paula Drive, Arkansas, N Little Rock",US
5285600,S3-257390113,Legacy Piedmont,"Portland, Maine, Forest Park",US
5285601,S3-97881155,Pranava Meribl Trust,"235, 13Th Cross Road, Hoyasala Ngr 2D Stage, B...",India


In [ ]:
# Check schema, shapes, and null values for all sources
for name, df in zip(['Source 1', 'Source 2', 'Source 3'], [df1, df2, df3]):
    print(f"=== {name} Shape: {df.shape} ===")
    print(df.isnull().mean() * 100)  # Percentage of missing values in each column
    print("-" * 40)

=== Source 1 Shape: (2206821, 4) ===
entity_id           0.0
business_name       0.0
business_address    0.0
country             0.0
dtype: float64
----------------------------------------
=== Source 2 Shape: (5034616, 4) ===
entity_id           0.000000
business_name       0.000040
business_address    3.356105
country             0.000000
dtype: float64
----------------------------------------
=== Source 3 Shape: (5285603, 4) ===
entity_id           0.000000
business_name       0.000246
business_address    3.328211
country             0.000000
dtype: float64
----------------------------------------


In [ ]:

gt

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
...,...,...
2206816,S1-824289006,S3-141788999
2206817,S1-856943494,"S2-950831962,S3-34095479,S3-718748776,S3-44048..."
2206818,S1-863031412,"S2-518155942,S2-631615245,S3-846786490,S3-1665..."
2206819,S1-711234676,"S2-908396708,S3-279383973,S3-969901413"


In [ ]:
# 1. Expand the comma-separated IDs into individual pairs
gt_exploded = gt.copy()
gt_exploded['matched_id'] = gt_exploded['matched_entity_ids'].str.split(',')
gt_exploded = gt_exploded.explode('matched_id')

# 2. Extract the source prefix (S2 or S3)
gt_exploded['matched_source'] = gt_exploded['matched_id'].str.extract(r'^(S[1-3])')

# 3. Calculate cluster size (how many total matches each S1 entity has)
cluster_sizes = gt['matched_entity_ids'].apply(lambda x: len(str(x).split(',')) if pd.notnull(x) else 0)

# 4. Print Summary Diagnostics
print("=== Ground Truth Analysis ===")
print(f"Total Source 1 Anchor Entities: {len(gt):,}")
print(f"Total Pairwise Matches (Exploded): {len(gt_exploded):,}")
print("\n--- Match Count Distribution per Source 1 Entity ---")
print(cluster_sizes.value_counts().head(10).sort_index())
print(f"Max matches for a single entity: {cluster_sizes.max()}")
print(f"Average matches per S1 entity: {cluster_sizes.mean():.2f}")

print("\n--- Match Breakdown by Target Dataset ---")
print(gt_exploded['matched_source'].value_counts())

#   What you see: Total count of links pointing to $S2$ vs $S3$.
#   What it means:Shows whether $S2$ and $S3$ contribute equally to
#   the ground truth matches or if one source is over-represented.
#   Action for Pipeline: If $S2$ has significantly fewer matches than $S3$,
#   check if $S2$ requires different text normalization or blocking strategies
#   (e.g., handling non-English scripts, as seen in $S2$ rows with Hindi/Devanagari text).

=== Ground Truth Analysis ===
Total Source 1 Anchor Entities: 2,206,821
Total Pairwise Matches (Exploded): 7,761,612

--- Match Count Distribution per Source 1 Entity ---
matched_entity_ids
0    123247
1    119157
2    375212
3    530841
4    484115
5    321957
6    164868
7     63968
8     18680
9      4205
Name: count, dtype: int64
Max matches for a single entity: 11
Average matches per S1 entity: 3.46

--- Match Breakdown by Target Dataset ---
matched_source
S3    3944746
S2    3693619
Name: count, dtype: int64


What you see: A frequency table showing how many Source 1 entities have 1 match, 2 matches, 5 matches, or 10+ matches across $S2$ and $S3$.
If most entities have 1 match: The problem is largely a 1-to-1 pair-matching problem across datasets.

If many entities have 3+ matches: Multiple records in Source 2 and Source 3 represent the exact same entity (intra-source duplicates or redundant listings).



Action for Pipeline: Your post-processing step cannot simply take the single top-scoring match per $S1$ record. You need a probability threshold (e.g., all candidate pairs with a similarity score $> 0.85$) or a graph-based clustering algorithm (e.g., Connected Components) to group all duplicate records together.

In [ ]:
# Extract unique IDs strictly present in each dataset
s1_gt_ids = set(gt['source1_entity_id'])
s2_gt_ids = set(gt_exploded[gt_exploded['matched_source'] == 'S2']['matched_id'].dropna().unique())
s3_gt_ids = set(gt_exploded[gt_exploded['matched_source'] == 'S3']['matched_id'].dropna().unique())

print(f"S1 Coverage in GT: {len(s1_gt_ids) / len(df1) * 100:.2f}%")
print(f"S2 Coverage in GT: {len(s2_gt_ids) / len(df2) * 100:.2f}%")
print(f"S3 Coverage in GT: {len(s3_gt_ids) / len(df3) * 100:.2f}%")
#Tells you what percentage of entities in each dataset
#are "singletons" (entities that have no matches in other datasets).

S1 Coverage in GT: 100.00%
S2 Coverage in GT: 73.36%
S3 Coverage in GT: 74.63%


In [ ]:
# Check country distribution across sources
print("S1 Countries:\n", df1['country'].value_counts(dropna=False).head(5))
print("\nS2 Countries:\n", df2['country'].value_counts(dropna=False).head(5))
print("\nS3 Countries:\n", df3['country'].value_counts(dropna=False).head(5))

# Check string length distributions
for name, df in zip(['Source 1', 'Source 2', 'Source 3'], [df1, df2, df3]):
    lens = df['business_name'].fillna('').str.len()
    print(f"{name} Name Lengths -> Min: {lens.min()}, Median: {lens.median()}, Max: {lens.max()}")

S1 Countries:
 country
US       1323633
India     883188
Name: count, dtype: int64

S2 Countries:
 country
US       3016817
India    2017799
Name: count, dtype: int64

S3 Countries:
 country
US       3170056
India    2115547
Name: count, dtype: int64
Source 1 Name Lengths -> Min: 3, Median: 24.0, Max: 105
Source 2 Name Lengths -> Min: 0, Median: 25.0, Max: 104
Source 3 Name Lengths -> Min: 0, Median: 25.0, Max: 123


In [ ]:
# Define common corporate entity designators to strip out
CORP_SUFFIXES = r'\b(inc|llc|llp|pvt|ltd|corp|corporation|co|limited|private|company|group|holdings|services|solutions|partners)\b'

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove corporate suffixes/designations
    text = re.sub(CORP_SUFFIXES, '', text)

    # 3. Remove punctuation and special symbols (keep alphanumeric characters and single spaces)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 4. Collapse multiple spaces into a single clean space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function to all datasets
for df_name, df in zip(['df1', 'df2', 'df3'], [df1, df2, df3]):
    print(f"Cleaning business_name and business_address in {df_name}...")
    df['clean_business_name'] = df['business_name'].apply(clean_text)
    df['clean_business_address'] = df['business_address'].apply(clean_text)

Cleaning business_name and business_address in df1...
Cleaning business_name and business_address in df2...
Cleaning business_name and business_address in df3...


In [ ]:
# Inspect cleaned results for Source 2
df1[['business_name', 'clean_business_name', 'business_address', 'clean_business_address']].head(10)

,business_name,clean_business_name,business_address,clean_business_address
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,Custom Wealth Services LLC,custom wealth,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,Moore Bitwise Inc,moore bitwise,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


In [ ]:
# Inspect cleaned results for Source 2
df2[['business_name', 'clean_business_name', 'business_address', 'clean_business_address']].head(10)

,business_name,clean_business_name,business_address,clean_business_address
0,राम मार्केटिंग प्राइवेट लिमिटेड,,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",kh no 570 13 new delhi west delhi delhi
1,-- Holloway Peak Inc Seafood,holloway peak seafood,"105 ELM ST, MORGANTON, NC",105 elm st morganton nc
2,आदित्य प्रॉपर्टीज एलएलपी,,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",g 3 571 gulmohar colony bhopal madhya pradesh
3,Summit Inc,summit,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",greensboro nc 19 1 2 stardust trail
4,Delta Tetlecommunication Inc,delta tetlecommunication,"914 PIERPONT AVE, CLEVELAND, OH",914 pierpont ave cleveland oh
5,Lee and Lawson,lee and lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",1702 pine avenue city of menomonie wi
6,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,shivshakti vidyalaya vidyalaya overseas www sh...,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",h no 204 c road hoshiarpur punjab punjab
7,Shree Infracon Private Ltd,shree infracon,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",63 2275 7 alhind tower first floor jafferkhan ...
8,Chavira Platinum Chimera LLC,chavira platinum chimera,"282 SAXONY DRIVE, FTT MITCHELL, KY",282 saxony drive ftt mitchell ky
9,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,foundation excel agency,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",hn 753 e 1 bharat nagar 104 1 1 erandwane maha...


In [ ]:
# Inspect cleaned results for Source 2
df3[['business_name', 'clean_business_name', 'business_address', 'clean_business_address']].head(10)

,business_name,clean_business_name,business_address,clean_business_address
0,wilfordhancock.com,wilfordhancock com,"Mack Rd, Haltom City, Texas",mack rd haltom city texas
1,International South Consultants Private Ltd,international south consultants,NaN,
2,LLC Moncada Léarning Center,moncada l arning center,"5780 Fawn Ct, Fort Worth, Texas",5780 fawn ct fort worth texas
3,Moyna's Coffee,moyna s coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",1 ivanhoe ave po box 6009 cincinnati ohio
4,Pvt. EFS Print Ventures Ltd.,efs print ventures,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",door no 183 41st cross 22nd main 9th block jay...
5,LLC Hernandez Colonial Redwood,hernandez colonial redwood,"2260- Housecreek Trail, Unit 407, Raleigh, Nor...",2260 housecreek trail unit 407 raleigh north c...
6,Classic Equity Partners Group,classic equity,"6885 Catalpa Bluff Ln, PO Box 5799, Dickinson,...",6885 catalpa bluff ln po box 5799 dickinson texas
7,Ectolumdrex dba X+ Madison Inc,ectolumdrex dba x madison,"S03575 Cty Tk M, Town Of Buffalo, WI",s03575 cty tk m town of buffalo wi
8,Animal Hanisch Hospirlg,animal hanisch hospirlg,"##8 Willow Oak Lane, Fl. 0, Saint Louis, Missouri",8 willow oak lane fl 0 saint louis missouri
9,Gomez Optimal,gomez optimal,"343 Hempstead 161, Hope, Arkansas",343 hempstead 161 hope arkansas
